### Overall setup
1. Set up stable baselines to include RL offline baselines
2. Set up gymnasium with mujoco - use this to record videos from untrained and trained policies from stable baselines and the custom method
3. Set up minari and download offline dataset to train policies using stable baselines and the custom method

In [1]:
import os
import sys

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from gymnasium import spaces
# from rl_zoo3.train import train
# from stable_baselines3 import PPO
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import minari
from minari import DataCollector


torch.manual_seed(42)

/home/azm0269@auburn.edu/ducks/newgym/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# import gymnasium as gym
# import minari
# import torch

In [ ]:
## since each episode can be of varying length you must pad the squences
def collate_fn(batch):
    return {
        "id": torch.Tensor([x.id for x in batch]),
        "observations": torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(x.observations) for x in batch],
            batch_first=True
        ),
        "actions": torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(x.actions) for x in batch],
            batch_first=True
        ),
        "rewards": torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(x.rewards) for x in batch],
            batch_first=True
        ),
        "terminations": torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(x.terminations) for x in batch],
            batch_first=True
        ),
        "truncations": torch.nn.utils.rnn.pad_sequence(
            [torch.as_tensor(x.truncations) for x in batch],
            batch_first=True
        )
    }

In [4]:
### load dataset
minari_dataset = minari.load_dataset("mujoco/hopper/simple-v0")
dataloader = DataLoader(minari_dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)

env = minari_dataset.recover_environment()
observation_space = env.observation_space
action_space = env.action_space
assert isinstance(observation_space, spaces.Box)
assert isinstance(action_space, spaces.Box)

/home/azm0269@auburn.edu/ducks/newgym/lib/python3.10/site-packages/minari/dataset/minari_dataset.py:204: UserWarning: Installed mujoco version 3.3.7 does not meet the requirement ==3.2.3.
We recommend to install the required version with `pip install "mujoco==3.2.3"`
  warnings.warn(


In [13]:
next(iter(dataloader))

{'id': tensor([ 740., 1619., 1321.,  802., 1889.,   45.,  445.,  334., 1296.,  335.,
          755., 1000.,  847.,  392., 1063.,  638.,  984., 1924.,  208., 1508.,
         1901., 1878., 1819.,  835.,  963., 1446., 1698., 1627.,  327.,  259.,
         1865., 1684.,  964., 1300.,  927.,  772., 1241., 1782.,  934.,  467.,
          659.,  540.,  537.,  635.,  841., 1322., 1631.,  332., 1750.,  429.,
         1948., 1491., 1779., 1297., 1075., 1828., 1239., 1092.,  150.,  192.,
         1415.,  462.,  171., 1674.,   59., 1496.,  996.,   21.,  924., 1294.,
           10., 1160.,  525., 1731., 1189., 1523., 1862.,  586., 1039.,  116.,
           17.,  216.,  989., 1790., 1287., 1226., 1808., 1272., 1225., 1254.,
          530.,  787., 1630., 1342., 1919., 1787., 1860., 1874., 1318.,  784.,
          568., 1927.,  434.,  653., 1357.,  783.,  223.,  929., 1245., 1280.,
          277.,  348.,  642., 1668.,  974.,   61., 1601.,  418.,  701., 1423.,
           74.,   71.,  813., 1749.,  295.,  4